# Yandex ML Cup — Full Challenge Walkthrough

> A single, self-contained notebook covering **all three tracks** of the Yandex ML Cup ML challenge: the task statements, the winning approaches, and **runnable** demos of the core ideas. Every code cell executes top-to-bottom on Kaggle with no external data or GPU required.

| Track | Task | Core idea | Result |
|---|---|---|---|
| **A** | Adaptive discrete-puzzle solver | exact solver → beam → learned A* cascade | score **82** |
| **B** | Multi-camera novel-view synthesis | optical-flow + RIFE + LiDAR blend | high PSNR |
| **C** | School-question answering | small LLM + exact arithmetic router | fast & accurate |

**The unifying theme across all three:** *use an exact, structure-aware method wherever one exists, and fall back to a learned/generic method only for the rest.* You'll see the same pattern three times.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
print('Environment ready.')

Environment ready.


---
# Track A — Adaptive Discrete Puzzle Solver

## The task

Build **one universal algorithm** that, given **50 min to train** and **25 min to solve**, cracks *any* reversible discrete puzzle through a single `gym.py` API — including hidden puzzles it has never seen.

| Puzzle | Move type | Mechanic |
|---|---|---|
| `game_15_2d` — 15-puzzle | `SWAP` + `EMPTY` | slide tiles into a blank |
| `toggle_lights` — Lights Out | `TOGGLE` | pressing a cell flips its row + column |
| `cylinder_game` — Varykon cylinder | `ROTATE` | rotate rings of a colored cylinder |
| *hidden* | unknown | same API, unknown mechanics |

The solver talks to the environment **only** through `reset / valid_actions / step / is_solved / encode_state`. It is never told which puzzle it's playing — that's what makes it *adaptive*.

## Idea 1 — a learned value function `V(s)`

Train a small net `V(s)` to estimate **how many moves remain** to the solved state, then use it as the A* heuristic (`f = g + V`). Training data is free via **backward walks** from the solved state:

```
solved ─move▶ s1 (dist≈1) ─move▶ s2 (dist≈2) ─▶ ... ─▶ sk (dist≈k)
```

The net reads through `encode_state` (each cell → a fixed 15-number vector), so the *same weights* run on every puzzle regardless of grid size.

**But the biggest lesson:** a learned `V` is a *fallback*, never the star. Whenever an exact, structure-aware solver applies, it wins.

## Idea 2 — Lights Out is linear algebra over GF(2)  ▶ *runnable*

Each light is on (1) or off (0). Pressing a cell flips a fixed set of lights, and pressing twice = not pressing (mod 2), and order doesn't matter. So the whole puzzle is a linear system over the two-element field **GF(2)**:

$$A\mathbf{x} = \mathbf{b} \pmod 2$$

Solve with Gaussian elimination mod 2 — **exact and instant**. This scored **20/20** where beam search and A* both scored 0.

In [2]:
def solve_gf2(A, b):
    """Gaussian elimination over GF(2). Returns x with A @ x == b (mod 2)."""
    A = A.copy() % 2
    b = b.copy() % 2
    n_rows, n_cols = A.shape
    pivots, row = [], 0
    for col in range(n_cols):
        piv = next((r for r in range(row, n_rows) if A[r, col]), None)
        if piv is None:
            continue
        A[[row, piv]] = A[[piv, row]]
        b[[row, piv]] = b[[piv, row]]
        for r in range(n_rows):
            if r != row and A[r, col]:
                A[r] ^= A[row]           # XOR == subtraction mod 2
                b[r] ^= b[row]
        pivots.append(col)
        row += 1
    x = np.zeros(n_cols, dtype=np.int8)
    for r, col in enumerate(pivots):
        x[col] = b[r]
    return x

def build_lights_out_matrix(n):
    """n x n Lights Out: button i flips itself + up/down/left/right neighbours."""
    N = n * n
    A = np.zeros((N, N), dtype=np.int8)
    for r in range(n):
        for c in range(n):
            i = r * n + c
            A[i, i] = 1
            for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
                rr, cc = r+dr, c+dc
                if 0 <= rr < n and 0 <= cc < n:
                    A[i, rr*n+cc] = 1
    return A

# Demo: scramble a solved (all-off) board with random presses, then solve it back.
# (A 5x5 Lights Out matrix is singular, so we start from a guaranteed-solvable board.)
n = 5
A = build_lights_out_matrix(n)
rng = np.random.RandomState(0)
true_presses = rng.randint(0, 2, n*n).astype(np.int8)
board = (A @ true_presses) % 2                 # the resulting lit state to clear

x = solve_gf2(A, board)
final = (board + A @ x) % 2                     # apply the found presses
print('Lit cells to clear:', int(board.sum()))
print('Presses found:     ', int(x.sum()))
print('All lights off after solving:', bool((final == 0).all()))

Lit cells to clear: 13
Presses found:      13
All lights off after solving: True


## Idea 3 — generic beam search for everything else

For sliding-like and unknown puzzles, **beam search with a smooth *mismatch* score** (count of cells not in place) generalised best. Counter-intuitively, a sharp Manhattan + linear-conflict ranking *hurt* the beam — it collapsed diversity. A softer score kept the frontier varied and solved more.

## The final cascade — exact first, generic last

```
detect move structure via valid_actions / encode_state
         │
  pure TOGGLE?     ─ yes ─▶ GF(2) exact solver            (optimal, instant)
         │ no
  EMPTY + SWAP?    ─ yes ─▶ fast-array IDA* (Manhattan+LC)
         │                   └─ leftover budget ─▶ beam (mismatch)
         │ no (rotational / hidden)
         └───────────────▶ long beam ─▶ A* with learned V (fallback)
```

## What actually moved the score (63 → 82)

Most gains came from **fixing bugs and tuning budgets**, not new models:

| Fix | Score |
|---|---|
| Baseline (beam + simple RL ranker) | ~63 |
| beam ran *only* for sliding puzzles → made beam the fallback for **all** types | 63 → 70 |
| reverse-BFS table `max_depth=6` (covered ~0% of 30–100-step instances) → raised to `max_depth=30`, `max_states=500k` | 70 → **82** |

And a performance lesson: fast native transitions (list-swap + compact integer keys) took the 15-puzzle from **40k → 114k children/s**, which raised the solve rate from **1/8 → 6/8** at the *same* time budget. **Under a wall clock, states-explored-per-second is the real objective.**

---
# Track B — Novel-View / Frame Synthesis for a Camera Rig

## The task

Reconstruct what one camera on a self-driving car saw at an **intermediate moment**, given the surrounding frames in time plus a dense LiDAR cloud. **Metric: PSNR** vs. ground truth.

- 6-camera rig: `front, left_fwd, left_bwd, right_fwd, right_bwd, rear`
- Inputs: all cameras at **t0** and **t1** (2 s apart), a **target camera** + **target time** in between, camera intrinsics/extrinsics, and ~10M LiDAR points in world coords.
- Output: the target camera's image at the target time — a frame that was never recorded.

## The strategy — blend three independent estimates

| Estimate | Signal | Strong where |
|---|---|---|
| 1. Temporal optical-flow interpolation | same camera at t0 & t1 | smooth motion |
| 2. RIFE deep frame interpolation | t0 & t1 frames (neural) | non-linear motion |
| 3. LiDAR image-based rendering | 3D geometry + other cameras | parallax, occlusion |

Blend weights (tuned): `temporal=0.72`, `rife=0.42`, `lidar=0.30`.

## Estimate 1 — temporal optical-flow interpolation  ▶ *runnable*

The target is a fraction `α` between t0 and t1. If we know how each pixel *moves* from frame 0→1, we can slide it `α` of the way there and blend — instead of naively cross-fading (averaging), which ghosts every moving thing into a double image.

The demo below is a **pure camera pan**, so the true motion is known exactly. We use that known flow to isolate the *interpolation principle*: **warp-and-blend reconstructs the middle frame; a naive average ghosts it.** In the real task this displacement field is **estimated** per-pixel by DIS optical flow (and refined by RIFE) — that estimation is the hard part, but the warp-and-blend step is the same.

In [3]:
import cv2

# A wide TEXTURED canvas. Cropping it at different x-offsets is a pure camera pan
# (a global translation). Because we built it, the true motion is known exactly.
H, Wc = 120, 260
yy, xx = np.mgrid[0:H, 0:Wc]
canvas = (120 + 50*np.sin(xx/12.0) + 30*np.cos(yy/9.0)).astype(np.uint8)
canvas = np.stack([canvas, canvas, canvas], -1)
canvas[28:52, 60:92]   = (235, 70, 70)      # a red landmark
canvas[74:96, 150:186] = (70, 200, 130)     # a green landmark
canvas = np.ascontiguousarray(canvas)

w, S = 180, 24                              # window width; camera pans S px between t0 and t1
frame0 = canvas[:, 0:w].copy()             # view at t0
frame1 = canvas[:, S:S+w].copy()           # view at t1
truth  = canvas[:, S//2:S//2+w].copy()     # true middle frame (alpha = 0.5)

def remap(img, flow, scale):
    h, w = img.shape[:2]
    yy, xx = np.mgrid[0:h, 0:w].astype(np.float32)
    return cv2.remap(img, xx + scale*flow[...,0], yy + scale*flow[...,1],
                     cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)

# Known flow for a pure pan: constant +S from frame0->frame1, -S the other way.
# (In the real pipeline DIS optical flow ESTIMATES this per-pixel field.)
flow01 = np.zeros((H, w, 2), np.float32); flow01[..., 0] = S
flow10 = np.zeros((H, w, 2), np.float32); flow10[..., 0] = -S

def interpolate(img0, img1, alpha=0.5):
    w0 = remap(img0, flow01, alpha)         # push frame 0 forward by alpha*S
    w1 = remap(img1, flow10, 1 - alpha)     # push frame 1 backward by (1-alpha)*S
    return ((1 - alpha)*w0 + alpha*w1).astype(np.uint8)

pred_flow = interpolate(frame0, frame1, 0.5)
pred_avg  = ((frame0.astype(float) + frame1.astype(float)) / 2).astype(np.uint8)

def psnr(a, b):
    mse = np.mean((a.astype(float) - b.astype(float))**2)
    return 99.0 if mse == 0 else 10*np.log10(255**2 / mse)

# Evaluate on the interior (exclude the S-px border that panning reveals/hides).
roi = (slice(None), slice(S, w - S))
print('PSNR vs the true middle frame (interior region):')
print(f'  Naive average       : {psnr(pred_avg[roi],  truth[roi]):.2f} dB')
print(f'  Flow warp-and-blend : {psnr(pred_flow[roi], truth[roi]):.2f} dB   <- higher is better')

PSNR vs the true middle frame (interior region):
  Naive average       : 21.46 dB
  Flow warp-and-blend : 99.00 dB   <- higher is better


In [ ]:
fig, ax = plt.subplots(1, 5, figsize=(15, 3))
for a, im, t in zip(ax, [frame0, frame1, pred_avg, pred_flow, truth],
                    ['frame t0', 'frame t1', 'naive average', 'flow warp+blend', 'ground truth']):
    a.imshow(im); a.set_title(t); a.axis('off')
plt.tight_layout(); plt.show()
print('The naive average doubles/ghosts the landmarks; flow warp-and-blend reconstructs the true middle frame.')

## Estimates 2 & 3 — RIFE + LiDAR geometry

**RIFE** ([ECCV 2022](https://github.com/megvii-research/ECCV2022-RIFE)) is a neural net built for exactly "given frame 0 & 1, synthesise the frame at `α`." It handles non-linear/deforming motion that hand-tuned flow smears, and is *mixed in* (not used alone) to avoid its occasional hallucinations.

**LiDAR image-based rendering** adds true 3D geometry to fix parallax. The pipeline (illustrative — needs the real dataset):

```python
def world_to_camera(xyz, c2w):
    R, t = c2w[:3,:3], c2w[:3,3]
    return (xyz - t) @ R                 # R orthonormal -> R^{-1} == R^T

def project(xyz_world, c2w, K):
    cam = world_to_camera(xyz_world, c2w)
    uv  = (K @ cam.T).T
    return uv[:,0]/uv[:,2], uv[:,1]/uv[:,2], cam[:,2]   # u, v, depth

# 1. project LiDAR into target view    -> dense depth map (z-buffer, nearest wins)
# 2. back-project each target pixel     -> 3D world point
# 3. re-project into a source camera    -> sample its colour (bilinear)
# 4. blend all three estimates by confidence, write near-lossless JPEG (PSNR metric)
```

**Why blend instead of pick one?** PSNR punishes any badly-wrong region, and the three methods fail *differently* — flow smears fast motion, RIFE hallucinates texture, LiDAR has holes at grazing angles. A confidence-weighted average lets each cover the others' failures.

---
# Track C — LLM for School Questions

## The task

Answer school-level questions (in Russian) correctly, briefly, and to the point — offline in a Docker container under a tight compute/time budget.

**Model:** a small **Qwen3** checkpoint (~0.6B params: hidden=1024, 28 layers) served with **vLLM**.

## The core decision — small model + hybrid routing

A 0.6B model is fast but unreliable at arithmetic. So route each question:

```
question ─┬─ looks like pure arithmetic? ─ yes ─▶ exact evaluator (Fraction math)
          └─ no ──────────────────────────────▶ vLLM (Qwen3, greedy decoding)
```

Same philosophy as Track A: deterministic problems get a deterministic solver; the learned model handles the rest.

## The exact arithmetic path  ▶ *runnable*

`try_arithmetic` is deliberately **conservative** — it only fires when confident, else returns `None` and defers to the LLM. It evaluates with `fractions.Fraction` over a **whitelisted AST** (never `eval()`), so `1/3 + 1/6` returns the exact `1/2` and no arbitrary code can run.

In [5]:
import ast, re
from fractions import Fraction

def _eval_frac(node):
    if isinstance(node, ast.Expression):
        return _eval_frac(node.body)
    if isinstance(node, ast.Constant) and isinstance(node.value, int):
        return Fraction(node.value, 1)
    if isinstance(node, ast.Constant) and isinstance(node.value, float):
        return Fraction(str(node.value))
    if isinstance(node, ast.UnaryOp) and isinstance(node.op, ast.USub):
        return -_eval_frac(node.operand)
    if isinstance(node, ast.BinOp):
        a, b = _eval_frac(node.left), _eval_frac(node.right)
        if isinstance(node.op, ast.Add):  return a + b
        if isinstance(node.op, ast.Sub):  return a - b
        if isinstance(node.op, ast.Mult): return a * b
        if isinstance(node.op, ast.Div):  return a / b
    raise ValueError('unsupported expression')

def try_arithmetic(question):
    q = str(question).lower().strip()
    if len(q) > 140:
        return None
    if not any(w in q for w in ('сколько', 'вычисли', 'посчитай', 'найди значение', 'реши пример')):
        return None
    if not any(op in q for op in ('+', '-', '*', '/', ':')):
        return None
    cands = re.findall(r'[0-9][0-9\s\+\-\*/:\.,\(\)]{1,90}[0-9\)]', q)
    if not cands:
        return None
    expr = re.sub(r'\s+', '', max(cands, key=len).replace(',', '.').replace(':', '/'))
    if not re.fullmatch(r'[0-9\+\-\*/\.\(\)]+', expr):
        return None
    try:
        val = _eval_frac(ast.parse(expr, mode='eval'))
    except Exception:
        return None
    return f'Ответ: {val.numerator}.' if val.denominator == 1 else f'Ответ: {val.numerator}/{val.denominator}.'

for q in ['Вычисли 47 * 23', 'Сколько будет 1/3 + 1/6?', 'Посчитай (12 + 8) / 5', 'Кто написал Войну и мир?']:
    print(f'{q!r:38} -> {try_arithmetic(q)}')

'Вычисли 47 * 23'                      -> Ответ: 1081.
'Сколько будет 1/3 + 1/6?'             -> Ответ: 1/2.
'Посчитай (12 + 8) / 5'                -> None
'Кто написал Войну и мир?'             -> None


## The routing layer  ▶ *runnable (mock LLM)*

The dispatcher tries arithmetic first and batches everything else for one vLLM call. Here we plug in a tiny mock "LLM" so the routing logic runs standalone; the real config is shown below.

In [6]:
def mock_llm(prompt):
    facts = {'войну и мир': 'Ответ: Лев Толстой.',
             'столица франции': 'Ответ: Париж.'}
    for k, v in facts.items():
        if k in prompt.lower():
            return v
    return 'Ответ: (сгенерировано моделью).'

def answer_all(questions, llm=mock_llm):
    answers = [None]*len(questions)
    llm_idx = []
    for i, q in enumerate(questions):
        det = try_arithmetic(q)
        if det is not None:
            answers[i] = ('[exact] ', det)
        else:
            llm_idx.append(i)
    for i in llm_idx:                       # in the real system: one batched vLLM call
        answers[i] = ('[llm]   ', llm(questions[i]))
    return answers

qs = ['Вычисли 47 * 23', 'Кто написал Войну и мир?', 'Сколько будет 100 / 8?', 'Столица Франции?']
for q, (route, a) in zip(qs, answer_all(qs)):
    print(f'{route}{q!r:34} -> {a}')

[exact] 'Вычисли 47 * 23'                  -> Ответ: 1081.
[llm]   'Кто написал Войну и мир?'         -> Ответ: Лев Толстой.
[exact] 'Сколько будет 100 / 8?'           -> Ответ: 25/2.
[llm]   'Столица Франции?'                 -> Ответ: Париж.


## The vLLM path (runs in the container with a GPU)

```python
from vllm import LLM, SamplingParams

llm = LLM(model='/workspace/weights', dtype='bfloat16',
          gpu_memory_utilization=0.88, max_model_len=1024, trust_remote_code=True)
sampling = SamplingParams(temperature=0.0, top_p=1.0, max_tokens=192,
                          stop=['<|im_end|>', '<|endoftext|>'])
outputs = llm.generate([make_prompt(q) for q in llm_questions], sampling)
```

| Setting | Value | Why |
|---|---|---|
| `temperature` | 0.0 | greedy — deterministic, best for factual QA |
| `max_tokens` | 192 | answers are short; caps runtime |
| `max_model_len` | 1024 | small KV cache → more throughput |
| `gpu_memory_utilization` | 0.88 | pack the KV cache as full as is safe |

A system prompt forces the style (*correct, brief, no rambling*), and raw output is cleaned of `<think>` blocks and chat-template markers.

---
# Conclusion — one pattern, three tracks

All three winning solutions share the same backbone: **exact / structure-aware method first, learned or generic method as the fallback.**

| Track | Exact method | Learned / generic fallback |
|---|---|---|
| **A** puzzles | GF(2) solver for Lights Out | learned `V(s)` A* + beam search |
| **B** view synthesis | LiDAR projective geometry | optical flow + RIFE neural interp |
| **C** QA | Fraction arithmetic evaluator | Qwen3 LLM via vLLM |

Other cross-cutting lessons:
- **Bugs and budgets beat architecture** — Track A went 63→82 mostly by fixing a search-routing bug and a table-depth bug.
- **Under a wall clock, throughput is the objective** — 3× faster transitions solved 6× more puzzles at the same time limit.
- **Ensembles win on squared-error metrics** — blending three view-synthesis estimates beats any single one because their failure modes don't overlap.
- **Route deterministic sub-problems to deterministic solvers** — free, exact points that a small learned model would otherwise get wrong.

*This notebook is a self-contained writeup — the full runnable solutions (Docker, gym API, RIFE weights, vLLM) live in the challenge repo alongside it.*